# Dedekind DSL: Analyst Tier
Declarative table workflow with built-in data-quality cleanup for messy real-world inputs.

> **See also**: [Splink](https://github.com/moj-analytical-services/splink) (MoJ) — probabilistic record linkage / entity resolution at scale (Fellegi-Sunter + EM).
> Splink and dedekind are complementary: the quality combinators here produce the standardised input that Splink requires,
> and Splink's output `cluster_id` is a natural panel key for a subsequent `smart_join`.
> The match ladder in `smart_join` is designed to accommodate Splink-style probabilistic scoring as a future higher tier (see [#253](https://github.com/vincentk/dedekind/issues/253)).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

try:
    from dedekind import SetDef, smart_join, table
except ModuleNotFoundError:
    candidate_roots = [Path.cwd(), *Path.cwd().parents]
    for root in candidate_roots:
        candidate = root / "python"
        if (candidate / "dedekind").exists():
            sys.path.insert(0, str(candidate))
            break
    from dedekind import SetDef, smart_join, table

print("Imported Analyst combinators: table, smart_join, group_by/summarize, pivot/unpivot")
if plt is None:
    print("matplotlib not available: chart cells will print guidance instead of rendering charts")

ModuleNotFoundError: No module named 'dedekind'

## Part 1: Messy Source Tables
Start from intentionally noisy inputs (mixed case, symbols, missing values, malformed numbers).

In [ ]:
df_sales = pd.DataFrame(
    [
        {"date": "2026-01-05", "product_id": " P1 ", "region": "North", "units": "10", "revenue": "100"},
        {"date": "2026-01-07", "product_id": "p2", "region": "NORTH", "units": "5", "revenue": "$250"},
        {"date": "2026-01-09", "product_id": "p3", "region": "south", "units": 7, "revenue": "140.0"},
        {"date": "2026-02-03", "product_id": "p1", "region": " SOUTH ", "units": 8, "revenue": "80"},
        {"date": "2026-02-10", "product_id": "p2", "region": "north", "units": "4x", "revenue": "200"},
        {"date": "2026-02-11", "product_id": "p3", "region": "south", "units": 10, "revenue": "oops"},
    ]
)

df_products = pd.DataFrame(
    [
        {"product_id": "p1", "category": " hardware "},
        {"product_id": "p2", "category": "SOFTWARE"},
        {"product_id": "p3", "category": "Accessories"},
    ]
)

df_regions = pd.DataFrame(
    [
        {"region": "north", "segment": "Enterprise"},
        {"region": "south", "segment": "SMB"},
    ]
)

sales = table("sales", df_sales)
products = table("products", df_products)
regions = table("regions", df_regions)

display(df_sales.head())

## Part 2: Clean, Join, And Aggregate
Apply quality combinators before business transformations, then build the monthly category report.

Econometrics note: this join + month derivation step is effectively building a panel (entity keys observed across time periods).

In [ ]:
sales_clean = (
    sales
    .normalize_text(["product_id", "region"], lower=True, strip=True)
    .coerce_numeric(["units", "revenue"], strip_symbols=True)
    .fill_missing(units=0, revenue=0)
    .expect_domain("region", ["north", "south"], on_fail="raise")
)

products_clean = (
    products
    .normalize_text(["product_id", "category"], lower=True, strip=True)
    .expect_domain(
        "category",
        ["hardware", "software", "accessories"],
        on_fail="coerce",
        unknown_value="unknown",
    )
)

regions_clean = regions.normalize_text(["region"], lower=True, strip=True)

report_plan = (
    smart_join(sales_clean, products_clean)
    .smart_join(regions_clean)
    .derive(month=lambda t: pd.to_datetime(t["date"]).dt.to_period("M").astype(str))
    .group_by(["month", "category"] )
    .summarize(revenue=("revenue", "sum"), units=("units", "sum"))
    .order_by(["month", "category"])
)

summary = report_plan.realize()

print("logical plan")
print(report_plan.explain())

print("\nsummary")
display(summary)

## Part 3: Error Analysis - Quality Lift From Additional Trusted Tables
We compare broken data, basic cleanup, correlation imputation, and then an enriched stage that adds a high-quality reference table to improve repair accuracy.

In [ ]:
# Reference (trusted) target for error analysis
reference_pivot = pd.DataFrame(
    [
        {"month": "2026-01", "accessories": 140.0, "hardware": 100.0, "software": 250.0},
        {"month": "2026-02", "accessories": 200.0, "hardware": 80.0, "software": 200.0},
    ]
)

# Additional high-quality reference table (trusted source)
df_product_price_hq = pd.DataFrame(
    [
        {"product_id": "p1", "unit_price_hq": 10.0},
        {"product_id": "p2", "unit_price_hq": 50.0},
        {"product_id": "p3", "unit_price_hq": 20.0},
    ]
)
product_price_hq = table("product_price_hq", df_product_price_hq)

# Stage 0: broken baseline
broken_plan = (
    smart_join(sales, products)
    .smart_join(regions)
    .derive(month=lambda t: pd.to_datetime(t["date"]).dt.to_period("M").astype(str))
    .derive(revenue_num=lambda t: pd.to_numeric(t["revenue"], errors="coerce"))
    .fill_missing(revenue_num=0)
    .group_by(["month", "category"] )
    .summarize(revenue=("revenue_num", "sum"))
    .pivot(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0)
    .order_by("month")
)
broken_pivot = broken_plan.realize()

# Stage 1: gap-fill only
gapfilled_sales = (
    sales
    .normalize_text(["product_id", "region"], lower=True, strip=True)
    .coerce_numeric(["units", "revenue"], strip_symbols=True)
    .fill_missing(units=0, revenue=0)
)
gapfill_pivot = (
    smart_join(gapfilled_sales, products_clean)
    .smart_join(regions_clean)
    .derive(month=lambda t: pd.to_datetime(t["date"]).dt.to_period("M").astype(str))
    .group_by(["month", "category"] )
    .summarize(revenue=("revenue", "sum"))
    .pivot(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0)
    .order_by("month")
    .realize()
)

# Stage 2: correlation-based repair
sales_corr_input = (
    sales
    .normalize_text(["product_id", "region"], lower=True, strip=True)
    .coerce_numeric(["units", "revenue"], strip_symbols=True)
)
sales_for_corr = smart_join(sales_corr_input, products_clean).realize()

valid_for_ratio = sales_for_corr[(sales_for_corr["units"] > 0) & (sales_for_corr["revenue"] > 0)].copy()
category_ratio = valid_for_ratio.groupby("category").apply(
    lambda g: g["revenue"].sum() / g["units"].sum() if g["units"].sum() else 0.0
).to_dict()
global_ratio = valid_for_ratio["revenue"].sum() / valid_for_ratio["units"].sum()

sales_corr_repaired = sales_for_corr.copy()
sales_corr_repaired["revenue"] = sales_corr_repaired["revenue"].fillna(
    sales_corr_repaired["units"] * sales_corr_repaired["category"].map(category_ratio).fillna(global_ratio)
)
sales_corr_repaired["revenue"] = sales_corr_repaired["revenue"].fillna(0)

corr_pivot = (
    table("corr_repaired", sales_corr_repaired)
    .smart_join(regions_clean)
    .derive(month=lambda t: pd.to_datetime(t["date"]).dt.to_period("M").astype(str))
    .group_by(["month", "category"] )
    .summarize(revenue=("revenue", "sum"))
    .pivot(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0)
    .order_by("month")
    .realize()
)

# Stage 3: add one extra trusted table to lift quality further
sales_hq_repaired = (
    smart_join(sales_corr_input, products_clean)
    .smart_join(product_price_hq)
    .realize()
)

hq_missing_mask = sales_hq_repaired["revenue"].isna()
sales_hq_repaired["revenue"] = sales_hq_repaired["revenue"].fillna(
    sales_hq_repaired["units"] * sales_hq_repaired["unit_price_hq"]
)
sales_hq_repaired["revenue"] = sales_hq_repaired["revenue"].fillna(0)

hq_enriched_pivot = (
    table("hq_enriched", sales_hq_repaired)
    .smart_join(regions_clean)
    .derive(month=lambda t: pd.to_datetime(t["date"]).dt.to_period("M").astype(str))
    .group_by(["month", "category"] )
    .summarize(revenue=("revenue", "sum"))
    .pivot(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0)
    .order_by("month")
    .realize()
)

# Stage 4: cleaned baseline pipeline from Part 2
cleaned_pivot = (
    report_plan
    .pivot(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0)
    .order_by("month")
    .realize()
)

quality_issues = sales_clean.quality_report()
region_values = regions_clean.to_set("region").realize()

def _align_pivot(df):
    cols = ["month", "accessories", "hardware", "software"]
    aligned = df.copy()
    for c in cols:
        if c not in aligned.columns:
            aligned[c] = 0.0
    return aligned[cols].sort_values("month").reset_index(drop=True)

def _pivot_mae(candidate, reference):
    c = _align_pivot(candidate).set_index("month")
    r = _align_pivot(reference).set_index("month")
    return (c - r).abs().stack().mean()

broken_mae = _pivot_mae(broken_pivot, reference_pivot)
gapfill_mae = _pivot_mae(gapfill_pivot, reference_pivot)
corr_mae = _pivot_mae(corr_pivot, reference_pivot)
hq_enriched_mae = _pivot_mae(hq_enriched_pivot, reference_pivot)
cleaned_mae = _pivot_mae(cleaned_pivot, reference_pivot)

error_report = pd.DataFrame(
    [
        {"stage": "broken", "pivot_mae": broken_mae},
        {"stage": "gap_fill_only", "pivot_mae": gapfill_mae},
        {"stage": "corr_imputed", "pivot_mae": corr_mae},
        {"stage": "extra_hq_table", "pivot_mae": hq_enriched_mae},
        {"stage": "cleaned_pipeline", "pivot_mae": cleaned_mae},
    ]
)

print("broken-data pivot table")
display(_align_pivot(broken_pivot))

print("correlation-imputed pivot table")
display(_align_pivot(corr_pivot))

print("extra-table-enriched pivot table")
display(_align_pivot(hq_enriched_pivot))

print("cleaned baseline pivot table")
display(_align_pivot(cleaned_pivot))

print("quality interventions")
display(quality_issues)

print("error reduction report (lower is better)")
display(error_report)

if plt is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

    _align_pivot(broken_pivot).set_index("month").plot(kind="bar", ax=axes[0], title="Broken Pivot")
    _align_pivot(hq_enriched_pivot).set_index("month").plot(kind="bar", ax=axes[1], title="Extra-Table-Enriched Pivot")

    axes[0].set_ylabel("Revenue")
    axes[1].set_ylabel("Revenue")
    axes[0].legend(title="Category")
    axes[1].legend(title="Category")

    error_report.plot(x="stage", y="pivot_mae", kind="bar", legend=False, ax=axes[2], title="Pivot Error By Stage")
    axes[2].set_ylabel("MAE vs Reference")
    axes[2].tick_params(axis="x", rotation=30)

    plt.show()
else:
    print("Install matplotlib to render charts: pip install matplotlib")

print("realized region values:", region_values)

In [ ]:
broken_check = _align_pivot(broken_pivot)
gapfill_check = _align_pivot(gapfill_pivot)
corr_check = _align_pivot(corr_pivot)
hq_enriched_check = _align_pivot(hq_enriched_pivot)
cleaned_check = _align_pivot(cleaned_pivot)

expected_columns = ["month", "accessories", "hardware", "software"]
assert list(hq_enriched_check.columns) == expected_columns, hq_enriched_check.columns.tolist()

expected_hq_rows = [
    {"month": "2026-01", "accessories": 140.0, "hardware": 100.0, "software": 250.0},
    {"month": "2026-02", "accessories": 200.0, "hardware": 80.0, "software": 200.0},
]
assert hq_enriched_check.to_dict(orient="records") == expected_hq_rows

# Adding one trusted table should improve over correlation-only and cleaned baseline
assert broken_mae > gapfill_mae
assert gapfill_mae >= corr_mae
assert corr_mae >= hq_enriched_mae
assert cleaned_mae >= hq_enriched_mae

assert sorted(region_values) == ["north", "south"]
assert int(quality_issues["count"].sum()) >= 2

plan_text = report_plan.explain()
assert "normalize_text" in plan_text and "summarize" in plan_text

print("quality-lift verified: extra trusted table improves pivot quality")
print("notebook-03-ok")

## Part 4: Analyst Shim -> Formal Shim Translation (Ideated)

This notebook remains useful for exploratory data workflows, but the operational backlog
automation posture is moving toward a formal plan/apply/verify DSL.

The mapping idea:
- analyst quality combinators -> formal `derive` + `verify` rules
- smart joins -> formal `link` and `annotate` actions
- report tables -> formal `report` output artifacts
- heuristic repairs -> formal `causal_hypothesis` or `granger_candidate` edges until confirmed

In [1]:
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class FormalAction:
    kind: str
    target: str
    payload: Dict[str, Any]
    reason: str

analyst_pipeline = [
    "normalize_text(product_id, region)",
    "coerce_numeric(units, revenue)",
    "expect_domain(region in {north,south})",
    "smart_join(products)",
    "group_by(month, category)",
    "summarize(revenue=sum, units=sum)",
]

formal_translation: List[FormalAction] = [
    FormalAction(
        kind="derive",
        target="sales",
        payload={"ops": ["normalize_text", "coerce_numeric"]},
        reason="canonicalize noisy input before relational composition",
    ),
    FormalAction(
        kind="verify",
        target="sales",
        payload={"rule": "domain(region) subset {north,south}"},
        reason="quality gate before downstream aggregation",
    ),
    FormalAction(
        kind="link",
        target="#171",
        payload={"edge_kind": "correlates_with", "to": "#319"},
        reason="relational substrate and graph-theory cluster connection",
    ),
    FormalAction(
        kind="annotate",
        target="#123",
        payload={"note": "translation evidence from analyst workflow"},
        reason="audit trail for backlog governance",
    ),
    FormalAction(
        kind="report",
        target="monthly-category-summary",
        payload={"shape": ["month", "category", "revenue", "units"]},
        reason="materialized artifact for planning and review",
    ),
]

print("analyst pipeline steps:")
for step in analyst_pipeline:
    print("-", step)

print("\nformal translation actions:")
for action in formal_translation:
    print(action)

assert len(formal_translation) >= 5
assert any(a.kind == "verify" for a in formal_translation)
assert any(a.kind == "report" for a in formal_translation)

print("analyst-to-formal translation sketch verified")

analyst pipeline steps:
- normalize_text(product_id, region)
- coerce_numeric(units, revenue)
- expect_domain(region in {north,south})
- smart_join(products)
- group_by(month, category)
- summarize(revenue=sum, units=sum)

formal translation actions:
FormalAction(kind='derive', target='sales', payload={'ops': ['normalize_text', 'coerce_numeric']}, reason='canonicalize noisy input before relational composition')
FormalAction(kind='verify', target='sales', payload={'rule': 'domain(region) subset {north,south}'}, reason='quality gate before downstream aggregation')
FormalAction(kind='link', target='#171', payload={'edge_kind': 'correlates_with', 'to': '#319'}, reason='relational substrate and graph-theory cluster connection')
FormalAction(kind='annotate', target='#123', payload={'note': 'translation evidence from analyst workflow'}, reason='audit trail for backlog governance')
FormalAction(kind='report', target='monthly-category-summary', payload={'shape': ['month', 'category', 'revenue',